# M1 Agent原子循环 — 手写底层

目标：不调包，手写Agent循环，理解底层。

**链路**：Prompt → Context/History → Tool解析 → 执行 → 更新History → 输出

对比Transformer attention（workspace `attention.py`）：Agent的上下文不是Q/K/V，而是可管理的文本历史。

> 本 notebook 只做**讲解与演示**。唯一的实现是同目录的 `core.py`，
> 这里直接 import 使用，避免两份实现各自演化后对不上。


## 1. 底层解析：Agent循环结构

`core.py` 里的 `AgentLoop` 把循环拆成四步：

1. **初始化 History** —— 类比 Transformer 的 `input_ids`，但存的是文本历史
2. **生成** —— 真实场景接 LLM API；本例简化为直接读取 prompt
3. **工具调用判断** —— `_parse_tool_call` 手写解析 JSON，不用框架
4. **执行并写回 History** —— 工具结果进入上下文，供下一步使用


In [ ]:
# 1. 载入实现：Agent 循环在 core.py，这里只负责调用
from core import AgentLoop, calc, query_db

agent = AgentLoop({"calc": calc, "query_db": query_db})
print("tools:", list(agent.tools))

## 2. 运行测试：代码生成Agent场景

解析需求 → 调用工具 → 输出结果

**关键**：`AgentLoop._parse_tool_call` 只会识别 JSON 形式的工具调用。
自然语言（`"Calculate 3+5"`）解析不出来，会被原样回显 —— 这不是 bug，
是最小实现刻意留下的边界。想跑通链路，prompt 必须是合法 JSON。


In [ ]:
# 2. 触发一次真实工具调用
result = agent.run('{"name": "calc", "args": {"x": 3, "y": 5}}')
print("result:", result)
print("history:", agent.history)

## 3. 对比：不带工具调用时

同样的循环，输入不是 JSON 时不会触发工具，直接返回。
对比 Transformer 里「有没有 attention 不影响能不能前向」——
Agent 的工具分支是显式的、可跳过的。


In [ ]:
# 3. 对比：无工具调用路径
plain = AgentLoop({"calc": calc, "query_db": query_db})
print(plain.run("Calculate 3+5 and query db for user_1"))
print("history:", plain.history)

## 4. 调包决策（M7预告：主流框架）
- **不用LangChain**：当你要理解底层、定制行为、减少依赖时（本模块）
- **用LangChain**：当你需要多工具链、复杂Memory、标准化评估时
- **用AutoGen**：多Agent协作场景
- **用LlamaIndex**：数据检索/知识库场景


## 5. 扩展练习（打通链路）
1. 修改`core.py`，增加Memory持久化（JSON文件）
2. 添加`file_read` Tool，模拟数据分析Agent
3. 对比：用LangChain重写同一功能，观察差异
4. 进入M2：理解Token/Embedding底层（参考`llm-bpe-tokenizer`）
